# Claude Managed Agents · Lab: Build a support-ticket triage agent

**Goal:** Build an agent that ranks a queue of support tickets, using account data that only your code can provide.

**Estimated time:** 55 minutes  
**Before this:** the Managed Agents session

You'll work through one build from start to finish. By the end, you'll have written the part of an agent app that's new if you've only used the Messages API: the code that answers the agent when it asks your system for data.

## What you'll build

A support team has eight open tickets. Some sound urgent and aren't; one sounds minor and isn't. The agent reads the tickets inside Anthropic's sandbox, asks **your** CRM about each customer, ranks the queue, and writes the ranking to a file you download and check.

| Step | What you do | Time |
|---|---|---:|
| 0 · Preflight | Check your key, Python, and SDK | 5 min |
| 1 · Set up | Meet the data, create the agent and environment | 10 min |
| 2 · First run | Run the quickstart loop and watch it stop | 5 min |
| 3 · Answer the agent | Write the code that handles your custom tool | 20 min |
| 4 · Verify | Download the ranking and check it | 5 min |
| 5 · Follow up | Send a second task to the same session | 5 min |
| Power user | Add a tool that needs your approval | optional |
| 6 · Clean up | Archive your sessions | 2 min |

**You need:** Python 3.10+, JupyterLab, and an Anthropic API key with Managed Agents access. Keep `helpers.py` and `tickets.json` in the same folder as this notebook. For setup problems, see Troubleshooting in `README.md`.

## 0 · Preflight

Run the next cell and paste your API key into the box that appears. It starts with `sk-ant-`. The key stays in this notebook's memory and is never saved to the file.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("API key (starts with sk-ant-): ")

In [ ]:
import helpers

helpers.preflight()

**You know it worked when** every line starts with ✓. A ✗ line tells you what to fix. Fix it before moving on; nothing later works without it.

## 1 · Set up

### The tickets and the CRM

The tickets are what the agent will read. The CRM is a stand-in for your company's customer database. The agent can't see it: it lives in this notebook, and that's the point.

In [ ]:
import json
from pathlib import Path

tickets = json.loads(Path("tickets.json").read_text())
for t in tickets:
    print(t["id"], t["customer_id"], "-", t["subject"])

In [ ]:
CRM = {
    "C-101": {"name": "Northwind Logistics", "plan": "Growth",     "arr_usd": 18000,  "renewal_date": "2027-02-01"},
    "C-102": {"name": "Brightpath Health",   "plan": "Enterprise", "arr_usd": 240000, "renewal_date": "2027-03-15"},
    "C-103": {"name": "Pixel & Pine",        "plan": "Free",       "arr_usd": 0,      "renewal_date": None},
    "C-104": {"name": "Meridian Freight",    "plan": "Enterprise", "arr_usd": 310000, "renewal_date": "2026-10-07"},
    "C-105": {"name": "Oakline Studio",      "plan": "Growth",     "arr_usd": 12000,  "renewal_date": "2026-12-01"},
}

### Agent, environment, session

You'll create three things, at different times:

- **Agent:** the configuration. Model, system prompt, tools. Create it once and reuse it.
- **Environment:** the container template the agent runs in. Installed packages and network access. It holds none of your data.
- **Session:** one job. It runs an agent in a fresh container from an environment, with your files mounted into it. You'll start several sessions from one agent in this lab.

Your tickets get attached to each **session**, not to the environment.

### Create the agent

Read the `lookup_account` definition before you run the cell. It has a name, a description, and an input schema, and no code. Built-in tools like `bash` and `read` run inside Anthropic's sandbox. A custom tool runs in *your* code, which you'll write in step 3.

In [ ]:
from anthropic import Anthropic

client = Anthropic()

SYSTEM_PROMPT = (
    "You triage customer support tickets. The tickets are in "
    "/mnt/session/uploads/tickets.json. For every ticket, call lookup_account "
    "with its customer_id before deciding its priority. Rank all tickets from "
    "most to least urgent, weighing both the problem and the customer's account. "
    "Write the result to /mnt/session/outputs/triage.json as a list of objects "
    "with rank, ticket_id, and a one-sentence reason."
)

LOOKUP_ACCOUNT_TOOL = {
    "type": "custom",
    "name": "lookup_account",
    "description": (
        "Look up a customer's account in the company CRM. Returns the company "
        "name, plan (Free, Growth, or Enterprise), annual revenue in USD, and "
        "renewal date. Use this for every ticket before ranking it, since the "
        "ticket text alone doesn't show how important the customer is. Returns "
        "an error if the customer_id isn't in the CRM."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {"type": "string", "description": "The customer ID from the ticket, like C-101"}
        },
        "required": ["customer_id"],
    },
}

if "agent" not in globals():  # create once; re-running this cell reuses it
    agent = client.beta.agents.create(
        name="Ticket Triage",
        model="claude-opus-5-5",
        system=SYSTEM_PROMPT,
        tools=[{"type": "agent_toolset_20260401"}, LOOKUP_ACCOUNT_TOOL],
    )
print("agent:", agent.id)

In [ ]:
if "environment" not in globals():
    environment = client.beta.environments.create(
        name="triage-env",
        config={"type": "cloud", "networking": {"type": "limited"}},  # triage needs no internet
    )
if "tickets_file" not in globals():
    tickets_file = client.files.upload(file=Path("tickets.json"))

print("environment:", environment.id)
print("tickets file:", tickets_file.id)

In [ ]:
if "sessions" not in globals():
    sessions = []  # every session you start, so step 6 can archive them


def start_session(title, agent_id=None):
    session = client.beta.sessions.create(
        agent=agent_id or agent.id,
        environment_id=environment.id,
        title=title,
        resources=[{"type": "file", "file_id": tickets_file.id, "mount_path": "/tickets.json"}],
    )
    sessions.append(session)
    return session


print("start_session is ready")

**You know it worked when** you see IDs starting with `agent_`, `env_`, and `file_`, then `start_session is ready`. Inside each session, the tickets appear at `/mnt/session/uploads/tickets.json`, the path the system prompt points to.

## 2 · First run: watch it stop

The next cell uses the event loop from the quickstart: send a message, print events, stop when the session goes idle.

**Before you run it, predict:** the agent will want to call `lookup_account`, and nothing on Anthropic's side can run it. What happens?

In [ ]:
session = start_session("First run")

with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text": "Triage the support tickets."}]}],
    )
    for event in stream:
        helpers.print_event(event)
        if event.type == "session.status_idle":
            break

print("\n\nSession went idle.")
print("stop_reason:", event.stop_reason.type)
print("waiting on:", event.stop_reason.event_ids)

### What just happened

The session didn't finish. It paused:

- It went **idle** with the stop reason `requires_action`.
- `event_ids` lists the `lookup_account` calls it's waiting on.

The agent asked your system for data, and the session will wait until your code answers. The quickstart loop treats every idle as "done," so it quit.

**You know it worked when** you see `requires_action` and a list of event IDs. This session stays stuck on purpose. You'll archive it in step 6 and start a fresh one in step 3.

## 3 · Answer the agent

> ✏️ **TRY IT**
>
> 1. Complete `lookup_account`. In step 1, your tool description promised the agent that it "returns an error if the customer_id isn't in the CRM." Keep that promise: return the customer's CRM record as a dict, or a dict with an `"error"` key when the customer isn't found. One ticket has an unknown customer.
> 2. Complete the `requires_action` branch of `run_turn`. For each call the session is waiting on, run the matching handler and build a `user.custom_tool_result` event. Then send the results with `send_results`.
>
> The rest of the loop is written for you. Reference: [Handling custom tool calls](https://platform.claude.com/docs/en/managed-agents/events-and-streaming).
>
> **Success:** the agent finishes, every `lookup_account` call gets exactly one answer, and the agent writes `triage.json`.

In [ ]:
# Step 3, task 1: see the TRY IT box above.
def lookup_account(customer_id):
    # TODO: return the customer's record from CRM.
    # TODO: when customer_id isn't in CRM, return a dict with an "error" key.
    return None


HANDLERS = {"lookup_account": lookup_account}  # tool name -> the function that answers it

In [ ]:
# Step 3, task 2: see the TRY IT box above. Your code goes in the requires_action branch.
def run_turn(session, text):
    """Send one message and handle events until the agent finishes. Returns call and result counts."""
    calls = {}  # custom tool calls seen so far, by event ID
    stats = {"calls": 0, "results_sent": 0}

    def send_results(results):
        client.beta.sessions.events.send(session.id, events=results)
        stats["results_sent"] += len(results)

    with client.beta.sessions.events.stream(session.id) as stream:
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        for event in stream:
            helpers.print_event(event)

            if event.type == "agent.custom_tool_use":
                calls[event.id] = event
                stats["calls"] += 1

            elif event.type == "session.status_idle":
                if event.stop_reason and event.stop_reason.type == "requires_action":
                    # The session is paused, waiting for your answers.
                    # event.stop_reason.event_ids lists the calls it's waiting on.
                    # TODO: for each waiting call, run its handler from HANDLERS,
                    #       build a user.custom_tool_result event, then send_results(...).
                    raise NotImplementedError("Complete the requires_action branch in run_turn")
                else:
                    print("\n\nAgent finished.")
                    break
    return stats

Run your code. Each run starts a fresh session with the same agent.

In [ ]:
%%time
session = start_session("Triage")
try:
    stats = run_turn(session, "Triage the support tickets.")
except NotImplementedError as e:
    stats = None
    print(f"\n\nstatus: {e}, then run this cell again")

In [ ]:
if globals().get("stats") is None:
    print("status: complete the TODOs, then run the cell above again")
else:
    print("lookup calls:", stats["calls"])
    print("results sent:", stats["results_sent"])
    if stats["calls"] == 0:
        print("✗ The agent made no lookups. Check the output above for what it did instead.")
    elif stats["results_sent"] == stats["calls"]:
        print("✓ Every call got exactly one answer")
    elif stats["results_sent"] > stats["calls"]:
        print("✗ Some calls were answered more than once. See hint 3.")
    else:
        print("✗ Some calls never got an answer.")

<details>
<summary><strong>Hint 1: Where are the calls?</strong></summary>

`event.stop_reason.event_ids` holds the IDs of the calls the session is waiting on. The calls themselves were saved in `calls` as each `agent.custom_tool_use` event arrived. So `calls[event_id].name` tells you which handler to use, and `calls[event_id].input` is a dict of its arguments.

</details>

<details>
<summary><strong>Hint 2: What does a result look like?</strong></summary>

```python
{
    "type": "user.custom_tool_result",
    "custom_tool_use_id": event_id,  # which call this answers
    "content": [{"type": "text", "text": json.dumps(result)}],
    "is_error": False,  # True when the lookup failed
}
```

`content` is a list of content blocks, like a message. JSON-encode the result so the agent can read every field.

</details>

<details>
<summary><strong>Hint 3: Answer each call once</strong></summary>

The session can pause more than once and list calls you've already answered, because it hasn't processed your earlier results yet. Keep a set of the IDs you've answered and skip them. Collect all the results for one pause and send them in a single `send_results` call.

</details>

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

A custom tool turns the agent's work into a conversation with your code. The agent emits a call, the session pauses, and your result becomes the next thing the agent reads. `is_error` tells the agent the lookup failed, so it treats the result as a failure rather than as account data. Returning an error for C-999, instead of inventing a record, lets the agent flag that ticket for a human.

```python
def lookup_account(customer_id):
    if customer_id not in CRM:
        return {"error": f"No account found for customer_id {customer_id}"}
    return CRM[customer_id]
```

Add `answered = set()` near the top of `run_turn`, then replace the `TODO` and the `raise` with:

```python
results = []
for event_id in event.stop_reason.event_ids:
    if event_id in answered:
        continue  # already answered; the session hadn't processed it yet
    call = calls[event_id]
    result = HANDLERS[call.name](**call.input)
    print(f"  -> answered {call.name}: {result}")
    results.append({
        "type": "user.custom_tool_result",
        "custom_tool_use_id": event_id,
        "content": [{"type": "text", "text": json.dumps(result)}],
        "is_error": "error" in result,
    })
    answered.add(event_id)
if results:
    send_results(results)
```

</details>

## 4 · Verify the output

The agent says it wrote `triage.json`. Check the file itself, not the agent's summary. Output files live in the sandbox at `/mnt/session/outputs/`; `helpers.download` copies one into this folder. It retries for a few seconds, since files can appear shortly after the agent finishes.

In [ ]:
triage_path = helpers.download(client, session.id, "triage.json")
triage = json.loads(triage_path.read_text())
for row in triage:
    print(row["rank"], row["ticket_id"], "-", row["reason"])

The tickets include three planted cases. Before you run the check, find them in the ranking above:

- **T-1004** sounds minor. Look at its account.
- **T-1003** is the loudest ticket. Look at its plan.
- **T-1002** is an Enterprise customer whose whole team is locked out.

In [ ]:
rank = {row["ticket_id"]: row["rank"] for row in triage}
checks = [
    (len(triage) == 8, "all 8 tickets ranked"),
    (rank.get("T-1002", 99) <= 2, "T-1002 (Enterprise, locked out) is in the top 2"),
    (rank.get("T-1004", 99) <= 3, "T-1004 (sounds minor; biggest account, renewing soon) is in the top 3"),
    (rank.get("T-1003", 0) >= 5, "T-1003 (all caps, Free plan, cosmetic) is 5th or lower"),
]
for passed, label in checks:
    print("✓" if passed else "✗", label)

**You know it worked when** all four lines show ✓. If T-1004 misses the top 3, look back at step 3: did C-104 get an answer? Its rank depends entirely on data from your tool.

The checks use ranges, not exact positions. The agent can reasonably order the top three differently between runs. What matters is that account data changed the ranking.

## 5 · Follow up in the same session

Send a second task to the same session. You won't resend the tickets or the account data.

**Predict:** will the agent call `lookup_account` again?

In [ ]:
%%time
followup = run_turn(
    session,
    "Draft a short reply to each of the top 3 tickets and save them to /mnt/session/outputs/replies.md",
)
print("\nlookups this turn:", followup["calls"])

In [ ]:
print(helpers.download(client, session.id, "replies.md").read_text())

**You know it worked when** `replies.md` prints. In our test run, the agent made no lookups on this turn. It drafted from what the session already held: the tickets, the account data, and its own ranking. With the Messages API, you'd resend all of that yourself.

## Power user (optional): a tool that needs your approval

*Finished early? Try this. Otherwise, skip to step 6.*

> ✏️ **TRY IT**
>
> Some actions shouldn't run just because an agent asked. Add a second custom tool, `escalate_ticket`, that pages the on-call engineer. Your handler asks you to approve each escalation before it "runs."
>
> 1. Define the tool: name `escalate_ticket`, inputs `ticket_id` and `reason`, and a description that says when to use it.
> 2. Write `escalate_ticket(ticket_id, reason)`. Show the request, ask `input("Approve? y/n ")`, and return a result that says whether it was approved.
> 3. Add it to `HANDLERS`. Create a second agent with both tools, so your step 1 agent stays unchanged. Start a session with it and call `run_turn`.
>
> **Success:** you're asked to approve at least one escalation. When you answer `n`, the agent reports that the escalation was declined, not that it happened.

In [ ]:
# Power user: see the TRY IT box above.
# TODO: define ESCALATE_TICKET_TOOL (LOOKUP_ACCOUNT_TOOL in step 1 shows the shape)
# TODO: write escalate_ticket(ticket_id, reason) and add it to HANDLERS
# TODO: create agent_v2 with both tools, start a session with
#       start_session("Triage with escalation", agent_id=agent_v2.id), and call run_turn

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

A declined escalation isn't an error, so the result has no `error` key and `is_error` stays false. The agent reads "declined" as an outcome to report. You control a custom tool's execution completely, so approval can live in your handler, the same place your real system would check permissions.

```python
ESCALATE_TICKET_TOOL = {
    "type": "custom",
    "name": "escalate_ticket",
    "description": (
        "Page the on-call engineer about a ticket that needs immediate engineering "
        "attention, such as an outage or errors affecting many requests. A human "
        "approves each escalation; the result says whether it was approved. Don't use "
        "it for billing, how-to, or cosmetic issues."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string", "description": "The ticket ID, like T-1002"},
            "reason": {"type": "string", "description": "One sentence on why this needs an engineer now"},
        },
        "required": ["ticket_id", "reason"],
    },
}


def escalate_ticket(ticket_id, reason):
    print(f"\n  Escalation requested for {ticket_id}: {reason}")
    approved = input("  Approve? y/n ").strip().lower() == "y"
    return {
        "ticket_id": ticket_id,
        "escalated": approved,
        "note": "Paged the on-call engineer" if approved else "Declined by a human reviewer",
    }


HANDLERS["escalate_ticket"] = escalate_ticket

if "agent_v2" not in globals():
    agent_v2 = client.beta.agents.create(
        name="Ticket Triage + Escalation",
        model="claude-opus-5-5",
        system=SYSTEM_PROMPT + " Use escalate_ticket for any ticket that needs immediate engineering attention.",
        tools=[{"type": "agent_toolset_20260401"}, LOOKUP_ACCOUNT_TOOL, ESCALATE_TICKET_TOOL],
    )

session_v2 = start_session("Triage with escalation", agent_id=agent_v2.id)
run_turn(session_v2, "Triage the tickets and escalate anything that needs an engineer right now.")
```

</details>

## 6 · Clean up

Archive every session you started. Archiving stops new events but keeps the history, so you can still read what happened.

The agent and environment are reusable. Archive them only when you're completely done (the commented lines). After that, no cell above will run again: you can't start a session in an archived environment.

In [ ]:
for s in sessions:
    try:
        client.beta.sessions.archive(s.id)
        print(s.id, client.beta.sessions.retrieve(s.id).status)
    except Exception as e:  # already archived, or still running
        print(s.id, "skipped:", e)

# Completely done? Uncomment these. Nothing above will run afterward.
# _ = client.beta.environments.archive(environment.id)
# _ = client.beta.agents.archive(agent.id)
# if "agent_v2" in globals():
#     _ = client.beta.agents.archive(agent_v2.id)

**You know it worked when** each session shows `terminated`, the status of an archived session.

## Recap

You built an agent app where the agent works in Anthropic's sandbox and asks your code for data it can't reach.

- **Agent, environment, session.** You created the agent and environment once and started several sessions from them. Your files were attached to each session.
- **A custom tool is a round trip.** The agent emits a call, the session pauses with `requires_action`, and your code sends a `user.custom_tool_result`. Your loop has to tell "paused, waiting on me" apart from "finished."
- **Sessions keep state.** The follow-up used the tickets, accounts, and ranking the session already held.

**Learn more**

- [Managed Agents overview](https://platform.claude.com/docs/en/managed-agents/overview): when to use it and how it's priced
- [Session event stream](https://platform.claude.com/docs/en/managed-agents/events-and-streaming): every event type, including custom tool calls
- [Files](https://platform.claude.com/docs/en/managed-agents/files): mounting inputs and downloading outputs
- [Session operations](https://platform.claude.com/docs/en/managed-agents/session-operations): retrieve, list, archive, and delete
- [Managed Agents cookbooks](https://github.com/anthropics/claude-cookbooks/tree/main/managed_agents): more complete examples